In [1]:
import pandas as pd
import sqlite3
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------
# Configuration (reproducible)
# -----------------------------
RANDOM_SEED = 7
N_PATIENTS = 6
N_PHARMACIES = 4
N_MEDICATIONS = 5
N_PROVIDERS = 3
N_EMR_ORDERS = 10
N_DISPENSES = 12

DB_PATH = Path("medication_data.db")
EXPORT_MATCHED_CSV = Path("matched_records.csv")

# If True, writes intermediate tables to SQLite for transparency.
WRITE_SQLITE_TABLES = True

# If True, exports only the matched dataset (reviewer expectation).
EXPORT_ONLY_MATCHED = True

# Simulating EMR Orders → Outside Pharmacy Dispense Linkage (Reproducible Demo)

This notebook simulates a pragmatic linkage workflow between **EMR medication orders** and **outside pharmacy dispense history** using **synthetic data** that can be run locally.

It is organized to mirror the manuscript steps:

1. **Define Study Population and Unit of Analysis**
2. **Define Key Concepts and Data Elements**
3. **Data Extraction** (simulated normalized source tables)
4. **Linkage Strategy** (tiered matching)
5. **Deduplication Across Data Sources**
6. **Pharmacy Normalization**
7. **Validation and Quality Assurance**

**Primary output (reviewer expectation):** a single matched dataset exported as `matched_records.csv`.

> Note: A PseudoSQL appendix is included to document additional fallback join options beyond what is executed in this demo.

In [7]:
# Step 1: Define Study Population and Unit of Analysis
# ------------------------------------------------------
# Unit of analysis: EMR medication order (one row per order).
# Dispense history records are linked to orders when possible.

# Step 2: Define Key Concepts and Data Elements
# --------------------------------------------
# Key linkage elements used in this demo:
# - Tier 1: Order_ID (when present in dispense feed)
# - Tier 2: Patient First/Last/DOB + Pharmacy Phone (normalized) + Medication identifier
#
# Additional fallback join options are documented in the PseudoSQL appendix.


def normalize_phone(value: object) -> str | None:
    """Normalize phone to digits-only; drop leading US country code if present."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = re.sub(r"\D+", "", str(value))
    if not s:
        return None
    # Handle leading country code '1' for 11-digit US numbers
    if len(s) == 11 and s.startswith("1"):
        s = s[1:]
    return s


def normalize_name(value: object) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip().upper()
    return s or None


def parse_dob(value: object) -> pd.Timestamp | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return None
    return ts.normalize()


# Step 3: Data Extraction (simulated normalized source tables)
# -----------------------------------------------------------
# We simulate normalized tables consistent with the ERD intent:
# Patient, Medication, Provider, Pharmacy, EMR_Medication_Orders, Dispense_History

patients = pd.DataFrame(
    [
        {"Patient_ID": "P001", "First_Name": "John", "Last_Name": "Doe", "Date_of_Birth": "1980-01-01"},
        {"Patient_ID": "P002", "First_Name": "Jane", "Last_Name": "Smith", "Date_of_Birth": "1975-05-12"},
        {"Patient_ID": "P003", "First_Name": "Alice", "Last_Name": "Johnson", "Date_of_Birth": "1990-07-23"},
        {"Patient_ID": "P004", "First_Name": "Bob", "Last_Name": "Brown", "Date_of_Birth": "1985-03-15"},
        {"Patient_ID": "P005", "First_Name": "Maria", "Last_Name": "Garcia", "Date_of_Birth": "1968-11-02"},
        {"Patient_ID": "P006", "First_Name": "Wei", "Last_Name": "Chen", "Date_of_Birth": "2001-09-09"},
    ]
)

medications = pd.DataFrame(
    [
        {"Medication_ID": "M001", "Generic_ID": "G001", "Generic_Name": "Lisinopril"},
        {"Medication_ID": "M002", "Generic_ID": "G002", "Generic_Name": "Metformin"},
        {"Medication_ID": "M003", "Generic_ID": "G003", "Generic_Name": "Atorvastatin"},
        {"Medication_ID": "M004", "Generic_ID": "G004", "Generic_Name": "Amoxicillin"},
        {"Medication_ID": "M005", "Generic_ID": "G005", "Generic_Name": "Albuterol"},
    ]
)

providers = pd.DataFrame(
    [
        {"Provider_ID": "PR001", "NPI": "1234567890"},
        {"Provider_ID": "PR002", "NPI": "2345678901"},
        {"Provider_ID": "PR003", "NPI": "3456789012"},
    ]
)

pharmacies = pd.DataFrame(
    [
        {"Pharmacy_ID": "PH001", "Pharmacy_Phone": "(555) 123-4567", "Pharmacy_Nabp": "NABP001"},
        {"Pharmacy_ID": "PH002", "Pharmacy_Phone": "555.234.5678", "Pharmacy_Nabp": "NABP002"},
        {"Pharmacy_ID": "PH003", "Pharmacy_Phone": "+1 555 345 6789", "Pharmacy_Nabp": "NABP003"},
        {"Pharmacy_ID": "PH004", "Pharmacy_Phone": "5554567890", "Pharmacy_Nabp": "NABP004"},
    ]
)

# Simulated EMR orders (unit of analysis)
emr_orders = pd.DataFrame(
    [
        {"Order_ID": "O1001", "Patient_ID": "P001", "Medication_ID": "M001", "Provider_ID": "PR001", "Pharmacy_ID": "PH001"},
        {"Order_ID": "O1002", "Patient_ID": "P002", "Medication_ID": "M002", "Provider_ID": "PR002", "Pharmacy_ID": "PH002"},
        {"Order_ID": "O1003", "Patient_ID": "P003", "Medication_ID": "M003", "Provider_ID": "PR003", "Pharmacy_ID": "PH003"},
        {"Order_ID": "O1004", "Patient_ID": "P004", "Medication_ID": "M004", "Provider_ID": "PR001", "Pharmacy_ID": "PH004"},
        {"Order_ID": "O1005", "Patient_ID": "P005", "Medication_ID": "M005", "Provider_ID": "PR002", "Pharmacy_ID": "PH001"},
        {"Order_ID": "O1006", "Patient_ID": "P006", "Medication_ID": "M001", "Provider_ID": "PR003", "Pharmacy_ID": "PH002"},
        # a couple more to create unmatched scenarios
        {"Order_ID": "O1007", "Patient_ID": "P001", "Medication_ID": "M002", "Provider_ID": "PR001", "Pharmacy_ID": "PH003"},
        {"Order_ID": "O1008", "Patient_ID": "P002", "Medication_ID": "M003", "Provider_ID": "PR002", "Pharmacy_ID": "PH004"},
        {"Order_ID": "O1009", "Patient_ID": "P003", "Medication_ID": "M004", "Provider_ID": "PR003", "Pharmacy_ID": "PH001"},
        {"Order_ID": "O1010", "Patient_ID": "P004", "Medication_ID": "M005", "Provider_ID": "PR001", "Pharmacy_ID": "PH002"},
    ]
)

# Simulated dispense history (outside pharmacy feed)
# Some rows include Order_ID (Tier 1 possible), others do not (Tier 2 needed).
# Include duplicates and phone-format variation to support Steps 5 and 6.
dispense_history = pd.DataFrame(
    [
        {"History_ID": "H2001", "Order_ID": "O1001", "Patient_ID": "P001", "Medication_ID": "M001", "Provider_ID": "PR001", "Pharmacy_ID": "PH001"},
        {"History_ID": "H2002", "Order_ID": None, "Patient_ID": "P002", "Medication_ID": "M002", "Provider_ID": "PR002", "Pharmacy_ID": "PH002"},
        {"History_ID": "H2003", "Order_ID": None, "Patient_ID": "P003", "Medication_ID": "M003", "Provider_ID": "PR003", "Pharmacy_ID": "PH003"},
        {"History_ID": "H2004", "Order_ID": "O1004", "Patient_ID": "P004", "Medication_ID": "M004", "Provider_ID": "PR001", "Pharmacy_ID": "PH004"},
        {"History_ID": "H2005", "Order_ID": None, "Patient_ID": "P005", "Medication_ID": "M005", "Provider_ID": "PR002", "Pharmacy_ID": "PH001"},
        # Duplicate dispense (same as H2002) to demonstrate dedup
        {"History_ID": "H2006", "Order_ID": None, "Patient_ID": "P002", "Medication_ID": "M002", "Provider_ID": "PR002", "Pharmacy_ID": "PH002"},
        # A dispense that should match by patient+dob+phone but has missing Patient_ID/Pharmacy_ID
        {"History_ID": "H2007", "Order_ID": None, "Patient_ID": None, "Medication_ID": "M001", "Provider_ID": "PR003", "Pharmacy_ID": None},
        # Unmatched dispense
        {"History_ID": "H2008", "Order_ID": None, "Patient_ID": "P999", "Medication_ID": "M001", "Provider_ID": "PR001", "Pharmacy_ID": "PH001"},
    ]
)

# Add patient name/DOB and pharmacy phone to dispense rows that are missing IDs (simulate real-world feeds)
dispense_history = dispense_history.merge(
    patients.rename(columns={"First_Name": "Patient_First_Name", "Last_Name": "Patient_Last_Name", "Date_of_Birth": "Patient_DOB"}),
    on="Patient_ID",
    how="left",
)

dispense_history = dispense_history.merge(
    pharmacies.rename(columns={"Pharmacy_Phone": "Pharmacy_Phone_Raw"}),
    on="Pharmacy_ID",
    how="left",
)

# Override H2007 with name/DOB/phone but missing IDs
mask_h2007 = dispense_history["History_ID"] == "H2007"
dispense_history.loc[mask_h2007, ["Patient_First_Name", "Patient_Last_Name", "Patient_DOB", "Pharmacy_Phone_Raw"]] = [
    "Wei",
    "Chen",
    "2001-09-09",
    "1 (555) 234-5678",  # matches PH002 after normalization
]

# Display simulated extracted tables
print("Patients:")
display(patients)
print("Pharmacies:")
display(pharmacies)
print("EMR Orders (unit of analysis):")
display(emr_orders)
print("Dispense History (outside feed):")
display(dispense_history[[
    "History_ID",
    "Order_ID",
    "Patient_ID",
    "Patient_First_Name",
    "Patient_Last_Name",
    "Patient_DOB",
    "Medication_ID",
    "Provider_ID",
    "Pharmacy_ID",
    "Pharmacy_Phone_Raw",
]])

Patients:


,Patient_ID,First_Name,Last_Name,Date_of_Birth
0,P001,John,Doe,1980-01-01
1,P002,Jane,Smith,1975-05-12
2,P003,Alice,Johnson,1990-07-23
3,P004,Bob,Brown,1985-03-15
4,P005,Maria,Garcia,1968-11-02
5,P006,Wei,Chen,2001-09-09


Pharmacies:


,Pharmacy_ID,Pharmacy_Phone,Pharmacy_Nabp
0,PH001,(555) 123-4567,NABP001
1,PH002,555.234.5678,NABP002
2,PH003,+1 555 345 6789,NABP003
3,PH004,5554567890,NABP004


EMR Orders (unit of analysis):


,Order_ID,Patient_ID,Medication_ID,Provider_ID,Pharmacy_ID
0,O1001,P001,M001,PR001,PH001
1,O1002,P002,M002,PR002,PH002
2,O1003,P003,M003,PR003,PH003
3,O1004,P004,M004,PR001,PH004
4,O1005,P005,M005,PR002,PH001
5,O1006,P006,M001,PR003,PH002
6,O1007,P001,M002,PR001,PH003
7,O1008,P002,M003,PR002,PH004
8,O1009,P003,M004,PR003,PH001
9,O1010,P004,M005,PR001,PH002


Dispense History (outside feed):


,History_ID,Order_ID,Patient_ID,Patient_First_Name,Patient_Last_Name,Patient_DOB,Medication_ID,Provider_ID,Pharmacy_ID,Pharmacy_Phone_Raw
0,H2001,O1001,P001,John,Doe,1980-01-01,M001,PR001,PH001,(555) 123-4567
1,H2002,None,P002,Jane,Smith,1975-05-12,M002,PR002,PH002,555.234.5678
2,H2003,None,P003,Alice,Johnson,1990-07-23,M003,PR003,PH003,+1 555 345 6789
3,H2004,O1004,P004,Bob,Brown,1985-03-15,M004,PR001,PH004,5554567890
4,H2005,None,P005,Maria,Garcia,1968-11-02,M005,PR002,PH001,(555) 123-4567
5,H2006,None,P002,Jane,Smith,1975-05-12,M002,PR002,PH002,555.234.5678
6,H2007,None,None,Wei,Chen,2001-09-09,M001,PR003,None,1 (555) 234-5678
7,H2008,None,P999,NaN,NaN,NaN,M001,PR001,PH001,(555) 123-4567


## Step 4: Linkage Strategy

We implement a **tiered linkage**:

- **Tier 1 (preferred):** exact `Order_ID` match (when present in both sources).
- **Tier 2 (fallback):** match on patient **first name + last name + DOB**, plus **normalized pharmacy phone**, plus medication identifier.

A PseudoSQL appendix later in the notebook documents additional fallback join options (e.g., Patient_ID-based composite keys, generic medication matching, NABP matching) even if not executed here.

In [8]:
# Step 6: Pharmacy Normalization
# -----------------------------
# Normalize pharmacy phone in both sources to enable robust matching.

pharmacies = pharmacies.copy()
pharmacies["Pharmacy_Phone_Normalized"] = pharmacies["Pharmacy_Phone"].apply(normalize_phone)

patients = patients.copy()
patients["First_Name_Normalized"] = patients["First_Name"].apply(normalize_name)
patients["Last_Name_Normalized"] = patients["Last_Name"].apply(normalize_name)
patients["DOB_Parsed"] = patients["Date_of_Birth"].apply(parse_dob)

# Build EMR order details (simulating extraction joins)
emr_details = (
    emr_orders
    .merge(patients[["Patient_ID", "First_Name", "Last_Name", "Date_of_Birth", "First_Name_Normalized", "Last_Name_Normalized", "DOB_Parsed"]], on="Patient_ID", how="left")
    .merge(medications, on="Medication_ID", how="left")
    .merge(providers, on="Provider_ID", how="left")
    .merge(pharmacies[["Pharmacy_ID", "Pharmacy_Phone", "Pharmacy_Phone_Normalized", "Pharmacy_Nabp"]], on="Pharmacy_ID", how="left")
)

# Build dispense details (already has patient name/DOB and pharmacy phone raw)
dispense_details = dispense_history.copy()
dispense_details["Patient_First_Name_Normalized"] = dispense_details["Patient_First_Name"].apply(normalize_name)
dispense_details["Patient_Last_Name_Normalized"] = dispense_details["Patient_Last_Name"].apply(normalize_name)
dispense_details["Patient_DOB_Parsed"] = dispense_details["Patient_DOB"].apply(parse_dob)
dispense_details["Pharmacy_Phone_Normalized"] = dispense_details["Pharmacy_Phone_Raw"].apply(normalize_phone)

# Step 5: Deduplication Across Data Sources
# ----------------------------------------
# Deduplicate dispense records on a pragmatic key (outside feeds often contain duplicates).
# Here we deduplicate on: patient name+DOB + medication + pharmacy phone + order_id (if present)
# keeping the first occurrence.

dedup_key_cols = [
    "Order_ID",
    "Patient_First_Name_Normalized",
    "Patient_Last_Name_Normalized",
    "Patient_DOB_Parsed",
    "Medication_ID",
    "Pharmacy_Phone_Normalized",
]

dispense_details_before = len(dispense_details)
dispense_details = dispense_details.drop_duplicates(subset=dedup_key_cols, keep="first").reset_index(drop=True)
dispense_details_after = len(dispense_details)

print(f"Dispense deduplication: {dispense_details_before} → {dispense_details_after} rows")

# Step 4: Linkage Strategy (execute)
# ---------------------------------
# Tier 1: Order_ID exact match
# Tier 2: Patient First/Last/DOB + Pharmacy Phone (normalized) + Medication_ID

# Candidate matches: Tier 1
cand_t1 = emr_details.merge(
    dispense_details,
    on="Order_ID",
    how="left",
    suffixes=("_EMR", "_DSP"),
)

cand_t1["Match_Type"] = cand_t1["History_ID"].apply(lambda x: "Matched by Order_ID" if pd.notna(x) else None)

# Candidate matches: Tier 2 (only for EMR orders not already matched in Tier 1)
matched_order_ids_t1 = set(cand_t1.loc[cand_t1["Match_Type"].notna(), "Order_ID"].unique())

emr_unmatched = emr_details.loc[~emr_details["Order_ID"].isin(matched_order_ids_t1)].copy()

cand_t2 = emr_unmatched.merge(
    dispense_details,
    left_on=[
        "First_Name_Normalized",
        "Last_Name_Normalized",
        "DOB_Parsed",
        "Medication_ID",
        "Pharmacy_Phone_Normalized",
    ],
    right_on=[
        "Patient_First_Name_Normalized",
        "Patient_Last_Name_Normalized",
        "Patient_DOB_Parsed",
        "Medication_ID",
        "Pharmacy_Phone_Normalized",
    ],
    how="left",
    suffixes=("_EMR", "_DSP"),
)

cand_t2["Match_Type"] = cand_t2["History_ID"].apply(
    lambda x: "Matched by Patient Name/DOB + Pharmacy Phone" if pd.notna(x) else "No Match"
)

# Combine results: Tier 1 matches (and non-matches) + Tier 2 for remaining
# For Tier 1, label non-matches as "No Match" (they will be replaced by Tier 2 results for unmatched orders)
res_t1 = cand_t1.copy()
res_t1.loc[res_t1["Match_Type"].isna(), "Match_Type"] = "No Match"

# Keep only Tier 1 rows for orders that matched in Tier 1
res_t1 = res_t1.loc[res_t1["Order_ID"].isin(matched_order_ids_t1)].copy()

matched_records = pd.concat([res_t1, cand_t2], ignore_index=True)

# Keep a clean, reviewer-facing matched dataset: only rows with a match
matched_only = matched_records.loc[matched_records["Match_Type"] != "No Match"].copy()

# Minimal output columns (can be expanded if manuscript requires)
output_cols = [
    "Order_ID",
    "History_ID",
    "Match_Type",
    "Patient_ID",
    "First_Name",
    "Last_Name",
    "Date_of_Birth",
    "Medication_ID",
    "Generic_Name",
    "Provider_ID",
    "NPI",
    "Pharmacy_ID",
    "Pharmacy_Phone",
    "Pharmacy_Phone_Normalized",
]

# Some columns come from EMR side; ensure they exist
for col in output_cols:
    if col not in matched_only.columns:
        matched_only[col] = None

matched_only = matched_only[output_cols]

print("Match counts by type:")
display(matched_only["Match_Type"].value_counts(dropna=False).to_frame("count"))

print("Matched dataset preview:")
display(matched_only)

Dispense deduplication: 8 → 7 rows
Match counts by type:


,count
Match_Type,
Matched by Patient Name/DOB + Pharmacy Phone,4
Matched by Order_ID,2


Matched dataset preview:


,Order_ID,History_ID,Match_Type,Patient_ID,First_Name,Last_Name,Date_of_Birth,Medication_ID,Generic_Name,Provider_ID,NPI,Pharmacy_ID,Pharmacy_Phone,Pharmacy_Phone_Normalized
0,O1001,H2001,Matched by Order_ID,None,John,Doe,1980-01-01,NaN,Lisinopril,None,1234567890,None,(555) 123-4567,NaN
1,O1004,H2004,Matched by Order_ID,None,Bob,Brown,1985-03-15,NaN,Amoxicillin,None,1234567890,None,5554567890,NaN
2,NaN,H2002,Matched by Patient Name/DOB + Pharmacy Phone,None,Jane,Smith,1975-05-12,M002,Metformin,None,2345678901,None,555.234.5678,5552345678
3,NaN,H2003,Matched by Patient Name/DOB + Pharmacy Phone,None,Alice,Johnson,1990-07-23,M003,Atorvastatin,None,3456789012,None,+1 555 345 6789,5553456789
4,NaN,H2005,Matched by Patient Name/DOB + Pharmacy Phone,None,Maria,Garcia,1968-11-02,M005,Albuterol,None,2345678901,None,(555) 123-4567,5551234567
5,NaN,H2007,Matched by Patient Name/DOB + Pharmacy Phone,None,Wei,Chen,2001-09-09,M001,Lisinopril,None,3456789012,None,555.234.5678,5552345678


## Step 7: Validation and Quality Assurance

This section provides lightweight QA checks to confirm the linkage behaved as expected:

- Dispense deduplication reduced duplicates.
- Phone normalization produced consistent keys.
- Match counts by tier are reported.
- The exported dataset contains only matched records (reviewer expectation).

In [4]:
# Optional: persist simulated tables to SQLite (transparency / reproducibility)
if WRITE_SQLITE_TABLES:
    conn = sqlite3.connect(DB_PATH)

    patients.to_sql("Patient", conn, if_exists="replace", index=False)
    medications.to_sql("Medication", conn, if_exists="replace", index=False)
    providers.to_sql("Provider", conn, if_exists="replace", index=False)
    pharmacies.to_sql("Pharmacy", conn, if_exists="replace", index=False)

    emr_orders.to_sql("EMR_Medication_Orders", conn, if_exists="replace", index=False)
    dispense_history.to_sql("Dispense_History", conn, if_exists="replace", index=False)

    emr_details.to_sql("EMR_Medication_Orders_Details", conn, if_exists="replace", index=False)
    dispense_details.to_sql("Dispense_History_Details", conn, if_exists="replace", index=False)

    conn.close()
    print(f"Wrote simulated tables to SQLite: {DB_PATH.resolve()}")
else:
    print("Skipping SQLite writes (WRITE_SQLITE_TABLES=False)")

Wrote simulated tables to SQLite: C:\Users\michaelsb\DispenseDataEHROrders\medication_data.db


## PseudoSQL Appendix: Join Strategy and Fallback Options (Documentation)

Below is a **documentation-only** PseudoSQL sketch showing common join strategies and fallback options.

> The executed demo in this notebook uses Tier 1 (`Order_ID`) and Tier 2 (patient name+DOB + pharmacy phone + medication). The additional options below are included to match the manuscript narrative and to show how teams often extend linkage logic.

```sql
-- Build EMR order details
select  o.Order_ID,
        p.Patient_ID, p.First_Name, p.Last_Name, p.Date_of_Birth,
        m.Medication_ID, m.Generic_ID, m.Generic_Name,
        pr.Provider_ID, pr.NPI,
        ph.Pharmacy_ID, ph.Pharmacy_Phone, ph.Pharmacy_Nabp
into #EMR_Medication_Orders_Details
from EMR_Medication_Orders o
left join Patient  p  on o.Patient_ID   = p.Patient_ID
left join Medication m on o.Medication_ID = m.Medication_ID
left join Provider pr  on o.Provider_ID  = pr.Provider_ID
left join Pharmacy ph  on o.Pharmacy_ID  = ph.Pharmacy_ID;

-- Build dispense history details
select  d.History_ID,
        d.Order_ID,
        p.Patient_ID, p.First_Name, p.Last_Name, p.Date_of_Birth,
        m.Medication_ID, m.Generic_ID, m.Generic_Name,
        pr.Provider_ID, pr.NPI,
        ph.Pharmacy_ID, ph.Pharmacy_Phone, ph.Pharmacy_Nabp
into #Dispense_History_Details
from Dispense_History d
left join Patient  p  on d.Patient_ID   = p.Patient_ID
left join Medication m on d.Medication_ID = m.Medication_ID
left join Provider pr  on d.Provider_ID  = pr.Provider_ID
left join Pharmacy ph  on d.Pharmacy_ID  = ph.Pharmacy_ID;

-- Linkage strategy with fallbacks
select  e.*, d.*
from #EMR_Medication_Orders_Details e
left join #Dispense_History_Details d
    -- Tier 1: Best case (shared Order_ID)
    on e.Order_ID = d.Order_ID

    -- Tier 2: Composite identifiers (when IDs are present)
    or (
        e.Patient_ID   = d.Patient_ID
        and e.Medication_ID = d.Medication_ID
        and e.NPI      = d.NPI
        and e.Pharmacy_ID = d.Pharmacy_ID
    )

    -- Tier 3: Patient fallback when Patient_ID missing in dispense feed
    or (
        e.First_Name = d.First_Name
        and e.Last_Name = d.Last_Name
        and e.Date_of_Birth = d.Date_of_Birth
        and e.Medication_ID = d.Medication_ID
        and e.Pharmacy_ID = d.Pharmacy_ID
    )

    -- Tier 4: Medication fallback when Medication_ID mismatches
    -- (use Generic_ID or Generic_Name depending on your data)
    or (
        e.Patient_ID = d.Patient_ID
        and e.Generic_ID = d.Generic_ID
        and e.NPI = d.NPI
        and e.Pharmacy_ID = d.Pharmacy_ID
    )

    -- Tier 5: Pharmacy fallback when Pharmacy_ID missing
    -- (use normalized phone and/or NABP)
    or (
        e.Patient_ID = d.Patient_ID
        and e.Medication_ID = d.Medication_ID
        and e.NPI = d.NPI
        and e.Pharmacy_Phone = d.Pharmacy_Phone
    )
    or (
        e.Patient_ID = d.Patient_ID
        and e.Medication_ID = d.Medication_ID
        and e.NPI = d.NPI
        and e.Pharmacy_Nabp = d.Pharmacy_Nabp
    );
```

Important implementation note: in production, these `OR` joins are typically implemented as **tiered candidate generation + priority ranking** to avoid ambiguous multi-matches.

In [9]:
# Export (reviewer expectation: matched only)
matched_only.to_csv(EXPORT_MATCHED_CSV, index=False)
print(f"Exported matched dataset to: {EXPORT_MATCHED_CSV.resolve()}")

Exported matched dataset to: C:\Users\michaelsb\DispenseDataEHROrders\matched_records.csv


## Export

Exports only the matched dataset (reviewer expectation).

In [7]:
# Connect to database and perform the matching query
def match_records(db_name='medication_data.db'):
    conn = sqlite3.connect(db_name)
    
    # Query similar to the original SQL logic
    query = """
    SELECT 
        emr.*,
        dh.History_ID,
        dh.Order_ID as Dispense_Order_ID,
        CASE 
            WHEN emr.Order_ID = dh.Order_ID THEN 'Matched by Order_ID'
            WHEN emr.Patient_ID = dh.Patient_ID 
                AND emr.Medication_ID = dh.Medication_ID 
                AND emr.NPI = dh.NPI 
                AND emr.Pharmacy_ID = dh.Pharmacy_ID THEN 'Matched by Patient/Med/Provider/Pharmacy'
            ELSE 'No Match'
        END as Match_Type
    FROM EMR_Medication_Orders_Details emr
    LEFT JOIN Dispense_History_Details dh 
        ON emr.Order_ID = dh.Order_ID
        OR (emr.Patient_ID = dh.Patient_ID 
            AND emr.Medication_ID = dh.Medication_ID 
            AND emr.NPI = dh.NPI 
            AND emr.Pharmacy_ID = dh.Pharmacy_ID)
    """
    
    df_matched = pd.read_sql_query(query, conn)
    conn.close()
    
    print("Matched Records:")
    display(df_matched)
    
    print("\nMatch Summary:")
    print(df_matched['Match_Type'].value_counts())
    
    return df_matched

# Run the matching
df_matched = match_records()

Matched Records:


,Order_ID,Patient_ID,Name,Date_of_Birth,Medication_ID,Generic_ID,Generic_Name,Provider_ID,NPI,Pharmacy_ID,Pharmacy_Phone,Pharmacy_Nabp,History_ID,Dispense_Order_ID,Match_Type
0,1001,101,John Smith,1980-05-15,5001,9001,Lisinopril,201,1234567890,301,555-0101,1234567,2001.0,1001.0,Matched by Order_ID
1,1002,102,Mary Johnson,1975-08-22,5002,9002,Metformin,202,2345678901,302,555-0102,2345678,2002.0,1002.0,Matched by Order_ID
2,1003,103,Robert Brown,1990-03-10,5003,9003,Atorvastatin,201,1234567890,301,555-0101,1234567,2003.0,NaN,Matched by Patient/Med/Provider/Pharmacy
3,1004,104,Patricia Davis,1965-11-30,5004,9004,Omeprazole,203,3456789012,303,555-0103,3456789,2004.0,1004.0,Matched by Order_ID
4,1005,105,Michael Wilson,1985-07-18,5005,9005,Amlodipine,202,2345678901,302,555-0102,2345678,NaN,NaN,No Match



Match Summary:
Match_Type
Matched by Order_ID                         3
Matched by Patient/Med/Provider/Pharmacy    1
No Match                                    1
Name: count, dtype: int64


## Optional: Persist to SQLite

Writes simulated source tables and derived “details” tables to SQLite for transparency and reproducibility.

In [5]:
# Quick QA summary
print("QA summary")
print("----------")
print(f"EMR orders: {len(emr_orders)}")
print(f"Dispense records (post-dedup): {len(dispense_details)}")
print(f"Matched records exported: {len(matched_only)}")

# Basic missingness checks for key linkage fields in dispense feed
qa = pd.DataFrame(
    {
        "dispense_missing_order_id": [dispense_details["Order_ID"].isna().mean()],
        "dispense_missing_patient_name": [
            (dispense_details["Patient_First_Name_Normalized"].isna() | dispense_details["Patient_Last_Name_Normalized"].isna()).mean()
        ],
        "dispense_missing_dob": [dispense_details["Patient_DOB_Parsed"].isna().mean()],
        "dispense_missing_pharmacy_phone": [dispense_details["Pharmacy_Phone_Normalized"].isna().mean()],
    }
)

display(qa)

QA summary
----------
EMR orders: 10
Dispense records (post-dedup): 7
Matched records exported: 3


,dispense_missing_order_id,dispense_missing_patient_name,dispense_missing_dob,dispense_missing_pharmacy_phone
0,0.714286,0.857143,0.857143,0.857143
